In [2]:
import psi4
import numpy as np

# Memory specification
psi4.set_memory(int(5e8))
numpy_memory = 2

# Set output file
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
    C    0.000000    0.000000    0.000000
    O    0.000000    0.000000    1.203000
    H    0.000000    0.934000   -0.582000
    H    0.000000   -0.934000   -0.582000
    """)


# Set computation options
psi4.set_options({'basis': '6-31g**', 'puream': 0})

wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'))
mints = psi4.core.MintsHelper(wfn.basisset())

S = np.asarray(mints.ao_overlap())

nbf = S.shape[0]
ndocc = wfn.nalpha()

print('Number of occupied orbitals: %3d' % (ndocc))
print('Number of basis functions: %3d' % (nbf))

Number of occupied orbitals:   8
Number of basis functions:  40


In [3]:
# Memory check for ERI tensor
I_size = (nbf**4) * 8.e-9
print('\nSize of the ERI tensor will be {:4.2f} GB.'.format(I_size))
if I_size > numpy_memory:
    psi4.core.clean()
    raise Exception("Estimated memory utilization (%4.2f GB) exceeds allotted memory \
                     limit of %4.2f GB." % (I_size, numpy_memory))

# Build ERI Tensor
I = np.asarray(mints.ao_eri())

T = np.asarray(mints.ao_kinetic())
V = np.asarray(mints.ao_potential())
H = T + V


Size of the ERI tensor will be 0.02 GB.


In [4]:
# Symmetric orthogonalization
# s = U.T @ S @ U
# X = S^-0.5 = U @ s^-0.5 @ U.T

s, U = np.linalg.eigh(S)
s_inv_sqrt = 1.0 / np.sqrt(s)
# Reset the inf values to 0
s_inv_sqrt[np.isinf(s_inv_sqrt)] = 0.0
X = U @ np.diag(s_inv_sqrt) @ U.T

In [5]:
def diis(e_list: list) -> np.ndarray:
    """
    Takes in a list of np.ndarray objects with matching shapes as error vectors.
    Construct the DIIS matrix and solve the Pulay equations for the coefficients.
    Returns a coefficient vector matching the arrays in the list.
    """
    size = len(e_list)

    B = np.full((size + 1, size + 1), -1.0)
    B[-1, -1] = 0.0

    flattened_e = [e.ravel() for e in e_list]
    e_array = np.vstack(flattened_e)

    # Compute pairwise dot products directly into B
    B[:size, :size] = e_array @ e_array.T

    rhs = np.zeros(size + 1)
    rhs[-1] = -1.0

    c = np.linalg.solve(B, rhs)
    # Return only the coefficients corresponding to the error vectors
    return c[:size]

In [6]:
MAX_ITER = 100
STARTUP_ITER = 5 # Number of normal SCF iterations before turning on DIIS
E_CONV = 1e-6
GRAD_MAX = 1e-6
GRAD_RMS = 1e-6

V_nn = mol.nuclear_repulsion_energy()

# Initial guess, no electron interaction
F = H
E0_last = np.inf

F_list = []
e_list = []

for i in range(1, MAX_ITER+1):
    if (i == MAX_ITER + 1):
        psi4.core.clean()
        raise Exception("Maximum number of SCF iterations exceeded.")
    
    if (i == STARTUP_ITER + 1):
        print ("DIIS turned on!")

    # 1. Diagonalize current Fock matrix F
    F_p = X.T @ F @ X
    e, C_p = np.linalg.eigh(F_p)

    # C_p.shape = (K: original AO, K: new AO)
    # Truncate C to only keep occupied columns
    C = X @ C_p
    C_docc = C[:, :ndocc]

    # 2. Compute new Density matrix D
    D = C_docc @ C_docc.T

    # 3. Compute energy
    E0 = np.sum(D * (H + F)) + V_nn
    dE = E0 - E0_last

    # 4. Compute new F and error vector and append to list
    J = np.einsum('pqrs,rs->pq', I, D, optimize=True)
    K = np.einsum('prqs,rs->pq', I, D, optimize=True)
    F = H + 2 * J - K

    F_list.append(F)

    err_vector = F @ D @ S - S @ D @ F
    e_list.append(err_vector)

    max_error = np.max(np.abs(err_vector))
    rms_error = np.sqrt(np.mean(err_vector**2))

    print('SCF Iteration %3d: | Energy = %4.16f | dE = % 1.5E | Max_E = % 1.5E | RMS_E = % 1.5E' % (i, E0, dE, max_error, rms_error))

    # 5. Convergence check
    if np.abs(dE) < E_CONV and max_error < GRAD_MAX and rms_error < GRAD_RMS:
        psi4.core.clean()
        print("SCF Converged!")
        break

    E0_last = E0
    D_last = D

    if i < STARTUP_ITER:
        pass
    else:
        # Use a slice of the history to limit DIIS memory
        # Avoid old iterations from polluting the subspace
        history = slice(-8, None) 
        coeffs = diis(e_list[history])
        
        # Extrapolate F
        F = np.einsum('i,ijk->jk', coeffs, np.array(F_list[history]))

print('Final RHF Energy: %.10f a.u.' % E0)
E0_psi4 = psi4.energy('SCF')
print('PSI4 RHF Energy: %.10f a.u.' % E0_psi4)

SCF Iteration   1: | Energy = -206.0105332661359796 | dE = -INF | Max_E =  7.52923E-01 | RMS_E =  9.97746E-02
SCF Iteration   2: | Energy = -84.4517945584843943 | dE =  1.21559E+02 | Max_E =  5.38913E-01 | RMS_E =  1.22951E-01
SCF Iteration   3: | Energy = -140.6478092449703752 | dE = -5.61960E+01 | Max_E =  8.12638E-01 | RMS_E =  1.21970E-01
SCF Iteration   4: | Energy = -92.2437730647624647 | dE =  4.84040E+01 | Max_E =  6.87836E-01 | RMS_E =  1.40141E-01
SCF Iteration   5: | Energy = -134.2402988124035517 | dE = -4.19965E+01 | Max_E =  8.06466E-01 | RMS_E =  9.78968E-02
DIIS turned on!
SCF Iteration   6: | Energy = -110.9111187430972336 | dE =  2.33292E+01 | Max_E =  2.03304E-01 | RMS_E =  2.75705E-02
SCF Iteration   7: | Energy = -113.9065846569900202 | dE = -2.99547E+00 | Max_E =  2.87718E-02 | RMS_E =  3.76838E-03
SCF Iteration   8: | Energy = -113.9536084873430752 | dE = -4.70238E-02 | Max_E =  8.63016E-03 | RMS_E =  1.05009E-03
SCF Iteration   9: | Energy = -113.900370603849893